# N-grams

Steps :
1. Read the dataset (csv)
2. Clean the dataset
3. Split the dataset into train/test
4. Tokenize the scripts
5. Generate the N-gram model with the train's dataset
6. Predict with the test's dataset

Scoring : Perplexity

## Benchmark :
- Launch multiple N-grams (from 1 to 5) on different lengths (100, 200, 300, ...)
- Calculate perplexity

| Modèle         | Length | BLEU   | ROUGE-L | BERTScore | Perplexity   |
|----------------|--------|--------|---------|-----------|--------------|
| **3-Gram**     | 50     | 0.0187 | 0.3115  | 0.3707    | 5905.45      |
| **3-Gram**     | 100    | 0.0225 | 0.3210  | 0.3618    | 5905.45      |
| **4-Gram**     | 50     | 0.0197 | 0.1332  | 0.3740    | 194665.01    |
| **4-Gram**     | 100    | 0.0224 | 0.1329  | 0.3659    | 194665.01    |

# Common code
## Import librairies

In [2]:
import pandas as pd
from collections import Counter, defaultdict
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/rickgao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/rickgao/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Load the dataset

In [3]:
df_all = pd.read_csv("MovieDataThread.csv")
df = df_all.sample(n=1000, random_state=42) # Remove this line to use the entire dataset
display(df["Script"].head())

37439    Okay, go ahead.\n My name is Sidney Westerfeld...
11898    1\n A plague has descended upon Earth.\n In on...
24628    Hi, Leo.\n Say, kid, if these flowers ain't | ...
26980    1\n Under the light\n Of the full moon\n Shimm...
35203    Spend all day with us.\n There are two--\n par...
Name: Script, dtype: object

## Custom tokenizer

In [4]:
import re

def script_tokenize(text):
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\n", " NEWLINE_TOKEN ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip().split()

def custom_tokenize(text):
    first_tokenize = script_tokenize(text)
    entokenized = " ".join(first_tokenize)

    second_tokenize = word_tokenize(entokenized)
    return second_tokenize

## Split the dataset into train/test

In [5]:
# Split the data into training and testing sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Number of tokens
train_strings = " ".join(train_df["Script"])
train_tokens = custom_tokenize(train_strings)
print("Number of tokens in the training set:",len(train_tokens))

vocab = set(train_tokens)
print("Vocabulary size:",len(vocab))

test_strings = " ".join(test_df["Script"])
test_strings = test_strings.lower()
test_tokens = custom_tokenize(test_strings)
print("Number of tokens in the test set:",len(test_tokens))

Number of tokens in the training set: 9190071
Vocabulary size: 106672
Number of tokens in the test set: 2347761


# Text generation
## Next-word model

In [6]:
from nltk.util import ngrams
import random

# Generate a next word prediction model based on n-grams
def generate_next_word_model(df, n=2, n_scripts=None):
    """
    Generate a next word prediction model based on n-grams
    :param df: DataFrame containing the scripts
    :param n: The size of the n-grams
    :return: A dictionary where keys are n-1 tuples of words and values are Counters of the next word
    """
    next_word_model = defaultdict(Counter)
    if n_scripts is not None:
        df = df.sample(n_scripts, random_state=1)

    for script in df:
        tokens = custom_tokenize(script)
        ngrams_list = ngrams(tokens, n)
        for ngram_ in ngrams_list:
            prefix = ngram_[:-1]
            next_word = ngram_[-1]
            next_word_model[prefix][next_word] += 1
    return next_word_model

## Predict next word (from model), randomized and with history

In [7]:
# Generate the next word prediction model
def predict_next_word_randomized(model, prefix, n=2, history=None):
    """
    Predict the next word based on the prefix using a randomized approach
    :param model: The n-gram model
    :param prefix: The prefix to predict the next word for
    :param n: The size of the n-grams
    :param history: A set to keep track of previously generated n-grams
    :return: The predicted next word
    """
    tokens = custom_tokenize(prefix)
    prefix = tuple(tokens[-(n-1):]) if len(tokens) >= (n-1) else tuple(tokens)

    if prefix in model:
        next_word_counter = model[prefix]
        total = sum(next_word_counter.values())
        words = list(next_word_counter.keys())
        weights = [count / total for count in next_word_counter.values()]
        
        for _ in range(5):
            choice = random.choices(words, weights=weights, k=1)[0]
            # Check if it was already generated with the same prefix
            if not history or prefix + (choice,) not in history:
                return choice
        return choice
    else:
        return "."

## Text generation

In [8]:
# Text generation function
def generate_text(model, seed, n=4, max_len=100):
    """
    Generate text based on the seed and the n-gram model
    :param model: The n-gram model
    :param seed: The seed text to start the generation
    :param n: The size of the n-grams
    :param max_len: The maximum length of the generated text
    :return: The generated text
    """
    generated = seed
    history = set()
    for _ in range(max_len):
        next_word = predict_next_word_randomized(model, generated, n=n, history=history)
        prefix_tokens = tuple(word_tokenize(generated)[-(n-1):])
        history.add(prefix_tokens + (next_word,))
        generated += " " + next_word
    return generated

## Example

In [9]:
seed = "So what's goin' on?"
n_scripts = 1000
model = generate_next_word_model(df_all["Script"], n=4, n_scripts=n_scripts)
print("Model generated with", n_scripts, "scripts.")
print("Generated text:")
result = generate_text(model, seed, n=4, max_len=100)
result = result.replace(" NEWLINE_TOKEN ", "\n")
print(result)

Model generated with 1000 scripts.
Generated text:
So what's goin' on?
Who cares ?
- What the hell
are you one of your evenings .
It 's OK , matey .
We sling some bills .
-Nursing fees .
And there 'll be a lieutenant .
- What 's it doing there ?
We 'll have to put it .
It does n't add up . Something 's wrong with me !
The Williams girl ?
She is so disciplined .
It is amazing to think about now , is n't it normal and
humane that he


# Scoring
## BLEU/ROUGE Scoring

## Benchmark (BLEU/ROUGE scores, bert score, perplexity)

### Code of BLEU/ROUGE score

In [10]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
import re
import string

def calculate_bleu(reference, candidate):
    """
    Calculate BLEU score (it measures the similarity between two sentences)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU score
    """
    smoothing_function = SmoothingFunction().method4
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=smoothing_function)
    return bleu_score

def calculate_rouge(reference, candidate):
    """
    Calculate ROUGE-L score (it calculates the longest common subsequence)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The ROUGE score
    """
    rouge = Rouge()
    scores = rouge.get_scores(candidate, reference)
    return scores[0]['rouge-l']['f']

def preprocess_text(text):
    """
    Preprocess the text by removing punctuation and converting to lowercase
    :param text: The input text
    :return: The preprocessed text
    """
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = text.lower()
    return text

def calculate_scores(reference, candidate):
    """
    Calculate BLEU and ROUGE scores for the given reference and candidate sentences
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU and ROUGE scores
    """
    # Preprocess the text
    reference = preprocess_text(reference)
    candidate = preprocess_text(candidate)

    reference_tokens = word_tokenize(reference)
    candidate_tokens = word_tokenize(candidate)

    bleu_score = calculate_bleu(reference_tokens, candidate_tokens)
    rouge_score = calculate_rouge(reference, candidate)

    return bleu_score, rouge_score

### Perplexity calculation

In [11]:
import math

def calculate_perplexity(test_tokens, ngram_probabilities, n):
    """Calculates the perplexity of a test corpus given n-gram probabilities.
    Args:
        test_tokens (list): List of tokens from the test corpus.
        ngram_probabilities (dict): Dictionary containing n-gram probabilities.
        n (int): The size of the n-grams.
    Returns:
        float: The perplexity of the test corpus.
    """

    log_probability_sum = 0
    ngram_count = 0

    for i in range(len(test_tokens) - n + 1):
        ngram = tuple(test_tokens[i:i+n])
        prefix, word = ngram[:-1], ngram[-1]

        if prefix in ngram_probabilities and word in ngram_probabilities[prefix]:
            prob = ngram_probabilities[prefix][word]
        else:
            prob = 1e-8

        log_probability_sum += math.log2(prob)
        ngram_count += 1

    average_log_probability = -log_probability_sum / ngram_count
    perplexity = math.pow(2, average_log_probability)

    return perplexity


### Generate and evaluate

In [16]:
from bert_score import BERTScorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
import math

# Constants
N_GRAMS = [3]
GENERATE_LIMIT_SIZES = [50, 100]
SCRIPT_LIMIT_TOKEN_SIZE = 25
scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

def convert_counts_to_probabilities(model):
    """Convert the counts in the model to probabilities.
    :param model: The n-gram model
    :return: A dictionary where keys are n-1 tuples of words and values are dictionaries of next word probabilities
    """
    prob_model = {}
    for prefix, counter in model.items():
        total = sum(counter.values())
        prob_model[prefix] = {word: count / total for word, count in counter.items()}
    return prob_model

def calculate_perplexity(tokens, prob_model, n):
    """Calculate the perplexity of a sequence of tokens given an n-gram probability model.
    :param tokens: The sequence of tokens
    :param prob_model: The n-gram probability model
    :param n: The size of the n-grams
    :return: The perplexity of the sequence
    """
    log_prob_sum, count = 0.0, 0
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i + n])
        prefix, word = ngram[:-1], ngram[-1]
        prob = prob_model.get(prefix, {}).get(word, 1e-8)
        log_prob_sum += math.log2(prob)
        count += 1
    if count == 0:
        return float("inf")
    return 2 ** (-log_prob_sum / count)

def generate_and_evaluate(test_df, n, max_len, prob_model):
    """Generate text and evaluate it using BLEU, ROUGE, and perplexity scores.
    :param test_df: The DataFrame containing the test scripts
    :param n: The size of the n-grams
    :param max_len: The maximum length of the generated text
    :param prob_model: The n-gram probability model
    :return: The BLEU, ROUGE, and perplexity scores
    """
    bleu_scores, rouge_scores, perplexities = [], [], []
    bert_f1, bert_precision, bert_recall = [], [], []

    for script in test_df["Script"]:
        tokens = custom_tokenize(script)
        seed_tokens = tokens[:SCRIPT_LIMIT_TOKEN_SIZE]
        seed_text = " ".join(seed_tokens)

        generated = generate_text(prob_model, seed_text, n=n, max_len=max_len)
        candidate = generated[len(seed_text):].strip()
        reference = " ".join(tokens)[len(seed_text):len(seed_text) + len(candidate)]

        bleu, rouge = calculate_scores(reference, candidate)
        P, R, F1 = scorer.score([candidate], [reference])
        f1_score = F1.mean().item()
        precision_score = P.mean().item()
        recall_score = R.mean().item()
        perplexity = calculate_perplexity(tokens, prob_model, n)

        bleu_scores.append(bleu)
        rouge_scores.append(rouge)
        perplexities.append(perplexity)

        bert_f1.append(f1_score)
        bert_precision.append(precision_score)
        bert_recall.append(recall_score)

        with open(f"{n}-gram-{max_len}-tokens.csv", "a") as f:
            f.write(f"{seed_text};{generated};{reference};{bleu};{rouge};{perplexity};{f1_score};{precision_score};{recall_score}\n")

    print(f"\n===== Average scores for n-gram {n} and max length {max_len} =====")
    print(f"BLEU:     {np.mean(bleu_scores):.4f}")
    print(f"ROUGE-L:  {np.mean(rouge_scores):.4f}")
    print(f"Perplex.: {np.mean(perplexities):.2f}")
    print(f"BERT F1:   {np.mean(bert_f1):.4f}")
    print(f"BERT P:    {np.mean(bert_precision):.4f}")
    print(f"BERT R:    {np.mean(bert_recall):.4f}")

    return bleu_scores, rouge_scores, perplexities, bert_f1, bert_precision, bert_recall


print("==" * 40)
print("Evaluation of the model...")
print("==" * 40)
for n in N_GRAMS:
    model = generate_next_word_model(train_df["Script"], n=n)
    prob_model = convert_counts_to_probabilities(model)

    for max_len in GENERATE_LIMIT_SIZES:
        generate_and_evaluate(test_df, n, max_len, prob_model)


Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.dense.bias', 'lm_head.decoder.weight', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Evaluation of the model...

===== Average scores for n-gram 3 and max length 50 =====
BLEU:     0.0196
ROUGE-L:  0.1327
Perplex.: 5905.45
BERT F1:   0.3753
BERT P:    0.3757
BERT R:    0.3741

===== Average scores for n-gram 3 and max length 100 =====
BLEU:     0.0231
ROUGE-L:  0.1305
Perplex.: 5905.45
BERT F1:   0.3611
BERT P:    0.3560
BERT R:    0.3655


In [17]:
from bert_score import BERTScorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
import math

# Constants
N_GRAMS = [4]
GENERATE_LIMIT_SIZES = [50, 100]
SCRIPT_LIMIT_TOKEN_SIZE = 25
scorer = BERTScorer(lang="en", model_type="roberta-base", rescale_with_baseline=True)

def convert_counts_to_probabilities(model):
    """Convert the counts in the model to probabilities.
    :param model: The n-gram model
    :return: A dictionary where keys are n-1 tuples of words and values are dictionaries of next word probabilities
    """
    prob_model = {}
    for prefix, counter in model.items():
        total = sum(counter.values())
        prob_model[prefix] = {word: count / total for word, count in counter.items()}
    return prob_model

def calculate_perplexity(tokens, prob_model, n):
    """Calculate the perplexity of a sequence of tokens given an n-gram probability model.
    :param tokens: The sequence of tokens
    :param prob_model: The n-gram probability model
    :param n: The size of the n-grams
    :return: The perplexity of the sequence
    """
    log_prob_sum, count = 0.0, 0
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i + n])
        prefix, word = ngram[:-1], ngram[-1]
        prob = prob_model.get(prefix, {}).get(word, 1e-8)
        log_prob_sum += math.log2(prob)
        count += 1
    if count == 0:
        return float("inf")
    return 2 ** (-log_prob_sum / count)

def generate_and_evaluate(test_df, n, max_len, prob_model):
    """Generate text and evaluate it using BLEU, ROUGE, and perplexity scores.
    :param test_df: The DataFrame containing the test scripts
    :param n: The size of the n-grams
    :param max_len: The maximum length of the generated text
    :param prob_model: The n-gram probability model
    :return: The BLEU, ROUGE, and perplexity scores
    """
    bleu_scores, rouge_scores, perplexities = [], [], []
    bert_f1, bert_precision, bert_recall = [], [], []

    for script in test_df["Script"]:
        tokens = custom_tokenize(script)
        seed_tokens = tokens[:SCRIPT_LIMIT_TOKEN_SIZE]
        seed_text = " ".join(seed_tokens)

        generated = generate_text(prob_model, seed_text, n=n, max_len=max_len)
        candidate = generated[len(seed_text):].strip()
        reference = " ".join(tokens)[len(seed_text):len(seed_text) + len(candidate)]

        bleu, rouge = calculate_scores(reference, candidate)
        P, R, F1 = scorer.score([candidate], [reference])
        f1_score = F1.mean().item()
        precision_score = P.mean().item()
        recall_score = R.mean().item()
        perplexity = calculate_perplexity(tokens, prob_model, n)

        bleu_scores.append(bleu)
        rouge_scores.append(rouge)
        perplexities.append(perplexity)

        bert_f1.append(f1_score)
        bert_precision.append(precision_score)
        bert_recall.append(recall_score)

        with open(f"{n}-gram-{max_len}-tokens.csv", "a") as f:
            f.write(f"{seed_text};{generated};{reference};{bleu};{rouge};{perplexity};{f1_score};{precision_score};{recall_score}\n")

    print(f"\n===== Average scores for n-gram {n} and max length {max_len} =====")
    print(f"BLEU:     {np.mean(bleu_scores):.4f}")
    print(f"ROUGE-L:  {np.mean(rouge_scores):.4f}")
    print(f"Perplex.: {np.mean(perplexities):.2f}")
    print(f"BERT F1:   {np.mean(bert_f1):.4f}")
    print(f"BERT P:    {np.mean(bert_precision):.4f}")
    print(f"BERT R:    {np.mean(bert_recall):.4f}")

    return bleu_scores, rouge_scores, perplexities, bert_f1, bert_precision, bert_recall


print("==" * 40)
print("Evaluation of the model...")
print("==" * 40)
for n in N_GRAMS:
    model = generate_next_word_model(train_df["Script"], n=n)
    prob_model = convert_counts_to_probabilities(model)

    for max_len in GENERATE_LIMIT_SIZES:
        generate_and_evaluate(test_df, n, max_len, prob_model)


Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.dense.bias', 'lm_head.decoder.weight', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Evaluation of the model...

===== Average scores for n-gram 4 and max length 50 =====
BLEU:     0.0219
ROUGE-L:  0.1333
Perplex.: 194665.01
BERT F1:   0.3774
BERT P:    0.3779
BERT R:    0.3761

===== Average scores for n-gram 4 and max length 100 =====
BLEU:     0.0220
ROUGE-L:  0.1310
Perplex.: 194665.01
BERT F1:   0.3651
BERT P:    0.3629
BERT R:    0.3664
